<a href="https://colab.research.google.com/github/Ericatuitui/MSSP607/blob/main/WeeklyModules/Week10/plotting_with_pandas_improved_by_Chatgpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plotting with Pandas — Improved Classroom Notebook

This revision follows the original lesson's sequence: **time series → relationships → distributions → counts and frequencies**. It keeps the pandas plotting examples while improving reliability, labeling, and interpretation.

**Datasets (not included in the uploaded notebook):** Facebook 2018 stock prices (`data/fb_stock_prices_2018.csv`) and USGS earthquake records (`data/earthquakes.csv`). Put the original course `data/` folder next to this notebook before running all cells.

**Learning goals:** use `DataFrame.plot()` and `Series.plot()`; choose an appropriate chart; customize `Axes`; avoid misleading interpretation of correlations, density plots, and tsunami flags.

## 1. Setup and data checks

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# Run from the course project folder containing the data/ directory.
DATA_DIR = Path('data')
required_files = [DATA_DIR / 'fb_stock_prices_2018.csv', DATA_DIR / 'earthquakes.csv']
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing data files: {missing}. Upload/extract the course data/ folder "
        "beside this notebook, or change DATA_DIR to its location."
    )

fb = pd.read_csv(required_files[0], index_col='date', parse_dates=['date']).sort_index()
quakes = pd.read_csv(required_files[1])
print('Facebook:', fb.shape, '| Earthquakes:', quakes.shape)
display(fb.head(3), quakes.head(3))

### Quick data-quality check
Before plotting, confirm column names and missing values. The later examples expect stock columns `open`, `high`, `low`, `close`, `volume` and earthquake columns `mag`, `magType`, `parsed_place`, `tsunami`, `time`.

In [ ]:
print('Stock columns:', fb.columns.tolist())
print('Earthquake columns:', quakes.columns.tolist())
required_stock = {'open', 'high', 'low', 'close', 'volume'}
required_quakes = {'mag', 'magType', 'parsed_place', 'tsunami', 'time'}
assert required_stock.issubset(fb.columns), 'Some expected stock columns are missing.'
assert required_quakes.issubset(quakes.columns), 'Some expected earthquake columns are missing.'
print('Missing values in key earthquake fields:')
display(quakes[list(sorted(required_quakes))].isna().sum().to_frame('missing_count'))

## Evolution over time
Line plots help us see how a variable changes over time. They are the default for the `kind` argument, but we can pass `kind='line'` to be explicit in our intent:

In [ ]:
fb.plot(
    kind='line',
    y='open',
    figsize=(10, 5),
    style='b-',
    legend=False,
    title='Evolution of Facebook Open Price'
)

We provided the `style` argument in the previous example; however, we can use the `color` and `linestyle` arguments to get the same result:

In [ ]:
fb.plot(
    kind='line',
    y='open',
    figsize=(10, 5),
    color='blue',
    linestyle='solid',
    legend=False,
    title='Evolution of Facebook Open Price'
)

We can also plot many lines at once by simply passing a list of the columns to plot:

In [ ]:
fb.iloc[:5,].plot(
    y=['open', 'high', 'low', 'close'],
    style=['b-o', 'r--', 'k:', 'g-.'],
    title='Facebook OHLC Prices during 1st Week of Trading 2018'
)

## Creating subplots
When plotting with pandas, creating subplots is simply a matter of passing `subplots=True` to the `plot()` method, and (optionally) specifying the `layout` in a tuple of `(rows, columns)`:

In [ ]:
# Choose only the numeric stock variables rather than plotting every numeric column.
stock_columns = ['open', 'high', 'low', 'close', 'volume']
axes = fb[stock_columns].plot(
    kind='line', subplots=True, layout=(3, 2), figsize=(13, 9),
    title='Facebook Stock Prices and Volume (2018)', legend=False
)
plt.tight_layout()
plt.show()

Note that we didn't provide a specific column to plot and pandas plotted all of them for us.

## Visualizing relationships between variables
### Scatter plots
We make scatter plots to help visualize the relationship between two variables. Creating scatter plots requires we pass in `kind='scatter'` along with a column for the x-axis and a column for the y-axis:

In [ ]:
fb.assign(
    max_abs_change=fb.high - fb.low
).plot(
    kind='scatter', x='volume', y='max_abs_change',
    title='Facebook Daily High - Low vs. Volume Traded'
)

The relationship doesn't seem to be linear, but we can try a log transform on the x-axis since the scales of the axes are very different. With pandas, we simply pass in `logx=True`:

In [ ]:
fb.assign(
    max_abs_change=fb.high - fb.low
).plot(
    kind='scatter', x='volume', y='max_abs_change',
    title='Facebook Daily High - Low vs. log(Volume Traded)',
    logx=True
)

With matplotlib, we could use `plt.xscale('log')` to do the same thing.

### Adding Transparency to Plots with `alpha`
Sometimes our plots have many overlapping values, but this can be impossible to see. This can be addressed by increasing the transparency of what we are plotting using the `alpha` parameter. It is a float on [0, 1] where 0 is completely transparent and 1 is completely opaque. By default this is 1, so let's put in a lower value and re-plot the scatter plot:

In [ ]:
fb.assign(
    max_abs_change=fb.high - fb.low
).plot(
    kind='scatter', x='volume', y='max_abs_change',
    title='Facebook Daily High - Low vs. log(Volume Traded)',
    logx=True, alpha=0.25
)

### Hexbins
In the previous example, we can start to see the overlaps, but it is still difficult. Hexbins are another plot type that divide up the plot into hexagons, which are shaded according to the density of points there. With pandas, this is the `hexbin` value for the `kind` argument. It can also be important to tweak the `gridsize`, which determines the approximate grid resolution:

In [ ]:
# Log requires positive volume values.
hex_data = fb.loc[fb['volume'] > 0].assign(
    log_volume=lambda x: np.log(x['volume']),
    max_abs_change=lambda x: x['high'] - x['low']
)
ax = hex_data.plot(
    kind='hexbin', x='log_volume', y='max_abs_change',
    title='Daily High–Low Range vs. Log(Volume)',
    cmap='Greys', gridsize=20, figsize=(8, 5)
)
ax.set_xlabel('Natural log of daily trading volume')
ax.set_ylabel('High − low ($)')
plt.show()

### Visualizing correlations with a heatmap
Pandas calculates correlations with `.corr()`; Matplotlib displays the resulting matrix. Explicit tick positions prevent mislabeled axes. **Correlation does not establish causation.**

In [ ]:
fb_corr = fb.assign(
    log_volume=lambda x: np.log(x['volume'].where(x['volume'] > 0)),
    max_abs_change=lambda x: x['high'] - x['low']
).select_dtypes(include='number').corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(fb_corr, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_xticks(range(len(fb_corr.columns)), labels=fb_corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(fb_corr.index)), labels=fb_corr.index)
ax.set_title('Correlation Matrix of Stock Variables')
fig.tight_layout()
plt.show()

In [ ]:
fb_corr.loc['max_abs_change', ['volume', 'log_volume']]

## Visualizing distributions
### Histograms
With the pandas `plot()` method, making histograms is as easy as passing in `kind='hist'`:

In [ ]:
fb.volume.plot(
    kind='hist',
    title='Histogram of Daily Volume Traded in Facebook Stock'
)
plt.xlabel('Volume traded') # label the x-axis (discussed in chapter 6)

We can overlap histograms to compare distributions provided we use the `alpha` parameter. For example, let's compare the usage and magnitude of the various `magTypes` in the data:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for magtype, group in quakes.groupby('magType'):
    magnitudes = group['mag'].dropna()
    if not magnitudes.empty:
        magnitudes.plot.hist(
            ax=ax, bins=25, alpha=0.35, label=f'{magtype} (n={len(magnitudes)})'
        )
ax.set(title='Earthquake Magnitude by Measurement Type', xlabel='Magnitude', ylabel='Count')
ax.legend(title='Magnitude type')
plt.tight_layout()
plt.show()

### Kernel density estimation (KDE)
`kind='kde'` estimates a **smooth probability density**, not the probability of one exact value. Probability corresponds to area under the density curve over an interval. This plot requires SciPy (usually available in common notebook environments).

In [ ]:
fb.high.plot(
    kind='kde',
    title='KDE of Daily High Price for Facebook Stock'
)
plt.xlabel('Price ($)') # label the x-axis (discussed in chapter 6)

### Adding to the result of `plot()`
The `plot()` method returns a matplotlib `Axes` object. We can store this for additional customization of the plot, or we can pass this into another call to `plot()` as the `ax` argument to add to the original plot.

It can often be helpful to view the KDE superimposed on top of the histogram, which can be achieved with this strategy:

In [ ]:
ax = fb.high.plot(kind='hist', density=True, alpha=0.5)
fb.high.plot(
    ax=ax, kind='kde', color='blue',
    title='Distribution of Facebook Stock\'s Daily High Price in 2018'
)
plt.xlabel('Price ($)') # label the x-axis (discussed in chapter 6)

### Empirical cumulative distribution function (ECDF)
An ECDF gives the proportion of observed values **less than or equal to** a threshold. We calculate it directly with NumPy to avoid requiring `statsmodels`.

In [ ]:
# An ECDF can be calculated with NumPy, without an extra dependency.
ml_magnitudes = quakes.loc[quakes['magType'].eq('ml'), 'mag'].dropna().to_numpy()
if ml_magnitudes.size == 0:
    raise ValueError('No non-missing ml magnitudes were found.')
x = np.sort(ml_magnitudes)
y = np.arange(1, len(x) + 1) / len(x)
fig, ax = plt.subplots(figsize=(8, 4))
ax.step(x, y, where='post')
ax.set(title='ECDF of Earthquake Magnitudes (ml)', xlabel='Magnitude', ylabel='Cumulative proportion')
ax.set_ylim(0, 1.02)
plt.show()

The ECDF at magnitude 3 estimates the **sample proportion** of `ml` earthquakes with magnitude ≤ 3. The percentage must be calculated from the actual dataset rather than assumed to equal 98%.

In [ ]:
# Calculate P(magnitude <= 3) from the data instead of hard-coding 98%.
threshold = 3
probability = (ml_magnitudes <= threshold).mean()
fig, ax = plt.subplots(figsize=(8, 4))
ax.step(x, y, where='post', label='ECDF')
ax.axvline(threshold, color='black', linestyle='--', alpha=0.6)
ax.axhline(probability, color='black', linestyle=':', alpha=0.6)
ax.scatter([threshold], [probability], color='black', zorder=3)
ax.set(title=f'P(ml magnitude ≤ {threshold}) = {probability:.1%}',
       xlabel='Magnitude', ylabel='Cumulative proportion', ylim=(0, 1.02))
ax.legend()
plt.show()

### Box plots
To make box plots with pandas, we pass `kind='box'` to the `plot()` method:

In [ ]:
fb.iloc[:,:4].plot(kind='box', title='Facebook OHLC Prices Boxplot')
plt.ylabel('price ($)') # label the x-axis (discussed in chapter 6)

This can also be combined with a `groupby()`:

In [ ]:
volume_groups = fb.assign(
    volume_bin=pd.cut(fb['volume'], bins=3, labels=['low', 'med', 'high'])
)
axes = volume_groups.boxplot(
    column=['open', 'high', 'low', 'close'], by='volume_bin',
    layout=(2, 2), figsize=(11, 7)
)
plt.suptitle('Facebook Prices by Trading Volume Group')
plt.tight_layout()
plt.show()

We can use this to see the distribution of magnitudes across the different measurement methods for earthquakes:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
quakes[['mag', 'magType']].dropna().boxplot(column='mag', by='magType', ax=ax)
ax.set(title='Earthquake Magnitudes by Measurement Type', xlabel='Magnitude type', ylabel='Magnitude')
fig.suptitle('')
plt.tight_layout()
plt.show()

## Counts and frequencies
### Bar charts
With pandas, we have the option of using the `kind` argument or using `plot.<kind>()`. Let's use `plot.bar()` here to show the evolution of monthly volume traded in Facebook stock over time:

In [ ]:
monthly_volume = fb.loc['2018-02':'2018-08', 'volume'].resample('MS').sum()
ax = monthly_volume.plot.bar(figsize=(9, 4), color='green', rot=0,
                             title='Monthly Facebook Trading Volume (Feb–Aug 2018)')
ax.set_xticklabels(monthly_volume.index.strftime('%b'))
ax.set(xlabel='Month', ylabel='Shares traded')
plt.tight_layout()
plt.show()

We can also change the orientation of the bars. Passing `kind='barh'` gives us horizontal bars instead of vertical ones. Let's use this to look at the top 15 places for earthquakes in our data:

In [ ]:
top_places = quakes['parsed_place'].value_counts().head(15)
ax = top_places.sort_values().plot.barh(
    figsize=(10, 6), title='Top 15 Earthquake Locations (Sep 18 – Oct 13, 2018)'
)
ax.set(xlabel='Number of earthquakes', ylabel='Location')
plt.tight_layout()
plt.show()

The `tsunami` field is a flag on earthquake records. Summing it counts **earthquake records marked as tsunami-related**, not necessarily distinct tsunami events.

In [ ]:
# This is a count of records flagged as tsunami-related, not a verified tsunami count.
tsunami_flags = (
    quakes.assign(tsunami_flag=quakes['tsunami'].eq(1).astype(int))
    .groupby('parsed_place')['tsunami_flag'].sum().nlargest(10)
)
ax = tsunami_flags.sort_values().plot.barh(
    figsize=(10, 5), title='Top Locations by Tsunami-Flagged Earthquake Records'
)
ax.set(xlabel='Flagged earthquake records', ylabel='Location')
plt.tight_layout()
plt.show()

Seeing that Indonesia is the top place for tsunamis during the time period we are looking at, we may want to look how many earthquakes and tsunamis Indonesia gets on a daily basis. We could show this as a line plot or with bars; since this section is about bars, we will use bars here:

In [ ]:
indonesia = quakes.loc[quakes['parsed_place'].eq('Indonesia')].copy()
# USGS event timestamps in this course file are milliseconds since Unix epoch.
indonesia['time'] = pd.to_datetime(indonesia['time'], unit='ms', utc=True)
indonesia['earthquakes'] = 1
indonesia['tsunami_flag'] = indonesia['tsunami'].eq(1).astype(int)
indonesia_daily = (
    indonesia.set_index('time')[['earthquakes', 'tsunami_flag']]
    .resample('D').sum()
)
ax = indonesia_daily.plot.bar(
    figsize=(13, 4), rot=45, title='Indonesia: Daily Earthquake Records and Tsunami Flags',
    xlabel='Date (UTC)', ylabel='Number of records'
)
ax.set_xticklabels(indonesia_daily.index.strftime('%b %d'))
ax.legend(['Earthquakes', 'Tsunami-flagged records'])
plt.tight_layout()
plt.show()

A vertical bar chart is useful when category labels are short:

In [ ]:
quakes.magType.value_counts().plot(
    kind='bar', title='Earthquakes Recorded per magType', rot=0
)

# label the axes (discussed in chapter 6)
plt.xlabel('magType')
plt.ylabel('earthquakes')

Compare magnitude-measurement methods across the **four locations with the most earthquake records** (calculated from this dataset rather than hard-coded).

In [ ]:
top_four = quakes['parsed_place'].value_counts().head(4).index
counts = (
    quakes.loc[quakes['parsed_place'].isin(top_four)]
    .groupby(['parsed_place', 'magType']).size().unstack(fill_value=0)
)
ax = counts.plot.bar(figsize=(10, 5), title='Magnitude Types in the Four Most Frequent Locations')
ax.set(xlabel='Location', ylabel='Number of earthquakes')
plt.tight_layout()
plt.show()

### Stacked bar chart

In [ ]:
pivot = quakes.assign(mag_bin=np.floor(quakes['mag'])).pivot_table(
    index='mag_bin', columns='magType', values='mag', aggfunc='count', fill_value=0
)
ax = pivot.plot.bar(stacked=True, rot=0, figsize=(10, 5),
                    title='Earthquake Counts by Integer Magnitude and Type')
ax.set(xlabel='Magnitude bin (floor)', ylabel='Number of earthquakes')
plt.tight_layout()
plt.show()

#### 100% stacked bars

Normalize each **magnitude-bin row** so that its measurement-type proportions add to 100%. These percentages describe the composition *within each magnitude bin*, not the proportion of each measurement type across all bins.

In [ ]:
# Divide each magnitude-bin row by its total to compare proportions.
row_totals = pivot.sum(axis=1).replace(0, np.nan)
normalized_pivot = pivot.div(row_totals, axis=0).fillna(0)
ax = normalized_pivot.plot.bar(
    stacked=True, rot=0, figsize=(10, 5),
    title='Magnitude-Type Composition Within Each Magnitude Bin'
)
ax.set(xlabel='Magnitude bin (floor)', ylabel='Proportion')
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.legend(title='Magnitude type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Summary: Choosing a chart

| Question | Chart | Important caution |
|---|---|---|
| How does a variable change over time? | Line | Check date order and units. |
| Are two variables associated? | Scatter / hexbin | Association is not causation. |
| How are values distributed? | Histogram / KDE / ECDF / box | KDE is a density; ECDF is cumulative proportion. |
| How many observations are in each category? | Bar | Counts depend on dataset coverage. |
| How do category compositions compare? | 100% stacked bar | Normalize by the intended denominator. |

### Practice (optional)
1. Plot Facebook's daily closing price and label both axes.
2. Find the **observed** fraction of `ml` earthquakes with magnitude ≤ 4.
3. Compare earthquake counts and proportions by `magType` and explain the difference.

**Note:** This notebook has not been executed here because the two source CSV files were not included in the upload. Its cells are designed to run in order after the files are provided.